# 10 — Start Mapping the North West Tribe Atlas, K=7

This notebook creates the first mapping outputs for the North West Atlas, including full North West maps, per-region map packs, and per-LAD/council map packs.

Important correction in this version:

- The heatmaps require the full profile CSVs because the `*_map_ready_*` files only contain dominant-cluster summary fields.
- Ward mapping therefore uses `k7_ward25_named_full_profile_v1_enriched.csv`.
- MSOA mapping therefore uses `k7_msoa21_named_full_profile_v1.csv`.

It still saves slim map outputs and joined GeoPackages at the end.

Maps produced:

1. Dominant cluster by Ward25
2. Dominant cluster by MSOA21
3. Dominant cluster share / confidence
4. Fragmentation index
5. Individual cluster-share heatmaps

The output maps are saved into:

`data/processed/maps_v1`


In [1]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

warnings.filterwarnings("ignore", category=UserWarning)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

## 1. Paths and configuration

Adjust the boundary paths manually if the automatic search does not find the correct files.

Expected project structure:

```text
Electoral_Tribes
├── data
│   ├── geography
│   │   └── boundaries
│   └── processed
│       ├── aggregations_v1
│       └── maps_v1
└── notebooks
```

In [2]:
NOTEBOOK_DIR = Path.cwd()

if NOTEBOOK_DIR.name.lower() == "notebooks":
    PROJECT_DIR = NOTEBOOK_DIR.parent
else:
    PROJECT_DIR = NOTEBOOK_DIR

DATA_DIR = PROJECT_DIR / "data"
GEOGRAPHY_DIR = DATA_DIR / "geography"

# Your boundary files are currently expected under data/geography.
# If you later create data/geography/boundaries, update this to that folder.
BOUNDARY_DIR = GEOGRAPHY_DIR

AGG_DIR = DATA_DIR / "processed" / "aggregations_v1"
MAP_DIR = DATA_DIR / "processed" / "maps_v1"

MAP_DIR.mkdir(parents=True, exist_ok=True)
BOUNDARY_DIR.mkdir(parents=True, exist_ok=True)

K = 7

# Use the full/named profiles for mapping, not the slim map-ready files.
# The slim map-ready files do not include the individual cluster-share columns needed for heatmaps.
WARD_CSV_PATH = AGG_DIR / "k7_ward25_named_full_profile_v1_enriched.csv"
MSOA_CSV_PATH = AGG_DIR / "k7_msoa21_named_full_profile_v1.csv"

# Fallbacks for older output names.
if not WARD_CSV_PATH.exists():
    fallback = AGG_DIR / "k7_ward25_full_profile_v1.csv"
    if fallback.exists():
        WARD_CSV_PATH = fallback

if not MSOA_CSV_PATH.exists():
    fallback = AGG_DIR / "k7_msoa21_full_profile_v1.csv"
    if fallback.exists():
        MSOA_CSV_PATH = fallback

OA_BASE_PATH = AGG_DIR / "k7_oa_geo_cluster_base_v1.csv"
CLUSTER_KEY_PATH = AGG_DIR / "k7_cluster_interpretation_key_v1.csv"

print("Project folder:", PROJECT_DIR)
print("Boundary folder:", BOUNDARY_DIR)
print("Aggregation folder:", AGG_DIR)
print("Map output folder:", MAP_DIR)

for path in [WARD_CSV_PATH, MSOA_CSV_PATH, OA_BASE_PATH]:
    print(path.name, "exists:", path.exists())
    if not path.exists():
        raise FileNotFoundError(path)


Project folder: c:\Users\keena\Documents\Electoral_Tribes
Boundary folder: c:\Users\keena\Documents\Electoral_Tribes\data\geography
Aggregation folder: c:\Users\keena\Documents\Electoral_Tribes\data\processed\aggregations_v1
Map output folder: c:\Users\keena\Documents\Electoral_Tribes\data\processed\maps_v1
k7_ward25_named_full_profile_v1_enriched.csv exists: True
k7_msoa21_named_full_profile_v1.csv exists: True
k7_oa_geo_cluster_base_v1.csv exists: True


## 2. Locate boundary files

The notebook attempts to find likely MSOA21 and Ward25 boundary files from `data/geography/boundaries`.

Supported formats: `.gpkg`, `.shp`, `.geojson`, `.json`.

If detection fails, set `MSOA_BOUNDARY_PATH` and `WARD_BOUNDARY_PATH` manually.

In [3]:
def find_boundary_file(boundary_dir: Path, include_terms, exclude_terms=None):
    exclude_terms = exclude_terms or []
    suffixes = {".gpkg", ".shp", ".geojson", ".json"}

    candidates = []
    for path in boundary_dir.rglob("*"):
        if path.suffix.lower() not in suffixes:
            continue

        name = path.name.lower()
        include_ok = all(term.lower() in name for term in include_terms)
        exclude_ok = not any(term.lower() in name for term in exclude_terms)

        if include_ok and exclude_ok:
            candidates.append(path)

    # Prefer GeoPackage if multiple candidates exist.
    candidates = sorted(candidates, key=lambda p: (p.suffix.lower() != ".gpkg", len(str(p))))
    return candidates

msoa_candidates = []
msoa_candidates += find_boundary_file(BOUNDARY_DIR, ["msoa", "2021"])
msoa_candidates += find_boundary_file(BOUNDARY_DIR, ["middle", "super", "2021"])

ward_candidates = []
ward_candidates += find_boundary_file(BOUNDARY_DIR, ["ward", "2025"])
ward_candidates += find_boundary_file(BOUNDARY_DIR, ["wd25"])
ward_candidates += find_boundary_file(BOUNDARY_DIR, ["wards", "may", "2025"])

# Deduplicate while preserving order.
def dedupe(paths):
    seen = set()
    out = []
    for p in paths:
        if p not in seen:
            out.append(p)
            seen.add(p)
    return out

msoa_candidates = dedupe(msoa_candidates)
ward_candidates = dedupe(ward_candidates)

print("MSOA boundary candidates:")
for p in msoa_candidates[:10]:
    print(" -", p)

print("\nWard boundary candidates:")
for p in ward_candidates[:10]:
    print(" -", p)

MSOA_BOUNDARY_PATH = BOUNDARY_DIR / "Middle_layer_Super_Output_Areas_December_2021_Boundaries_EW_BGC_V3_-1334546435986816930.gpkg"
WARD_BOUNDARY_PATH = BOUNDARY_DIR / "Wards_May_2025_Boundaries_UK_BGC.gpkg"

print("\nSelected MSOA boundary:", MSOA_BOUNDARY_PATH)
print("Selected Ward boundary:", WARD_BOUNDARY_PATH)

if MSOA_BOUNDARY_PATH is None:
    raise FileNotFoundError("Could not auto-detect MSOA boundary file. Set MSOA_BOUNDARY_PATH manually.")

if WARD_BOUNDARY_PATH is None:
    raise FileNotFoundError("Could not auto-detect Ward boundary file. Set WARD_BOUNDARY_PATH manually.")

MSOA boundary candidates:
 - c:\Users\keena\Documents\Electoral_Tribes\data\geography\Middle_layer_Super_Output_Areas_December_2021_Boundaries_EW_BGC_V3_-1334546435986816930.gpkg

Ward boundary candidates:
 - c:\Users\keena\Documents\Electoral_Tribes\data\geography\Wards_May_2025_Boundaries_UK_BGC.gpkg

Selected MSOA boundary: c:\Users\keena\Documents\Electoral_Tribes\data\geography\Middle_layer_Super_Output_Areas_December_2021_Boundaries_EW_BGC_V3_-1334546435986816930.gpkg
Selected Ward boundary: c:\Users\keena\Documents\Electoral_Tribes\data\geography\Wards_May_2025_Boundaries_UK_BGC.gpkg


## 3. Load full profile CSVs and boundary files

This version deliberately loads the full/named profile CSVs, not only the slim map-ready CSVs.

Reason: individual cluster-share heatmaps need columns such as `cluster_6_share` or `post_industrial_estates_deprived_working_communities_share`. The slim map-ready files usually do not include those columns.


In [4]:
ward_csv = pd.read_csv(WARD_CSV_PATH, low_memory=False)
msoa_csv = pd.read_csv(MSOA_CSV_PATH, low_memory=False)
oa_base = pd.read_csv(OA_BASE_PATH, low_memory=False)

ward_boundaries = gpd.read_file(WARD_BOUNDARY_PATH)
msoa_boundaries = gpd.read_file(MSOA_BOUNDARY_PATH)

print("Ward CSV:", ward_csv.shape, WARD_CSV_PATH.name)
print("MSOA CSV:", msoa_csv.shape, MSOA_CSV_PATH.name)
print("OA base:", oa_base.shape)
print("Ward boundaries:", ward_boundaries.shape)
print("MSOA boundaries:", msoa_boundaries.shape)

print("\nWard CSV columns:", ward_csv.columns.tolist())
print("\nMSOA CSV columns:", msoa_csv.columns.tolist())
print("\nWard boundary columns:", ward_boundaries.columns.tolist())
print("\nMSOA boundary columns:", msoa_boundaries.columns.tolist())

print("\nWard share-like columns:")
print([c for c in ward_csv.columns if "share" in c.lower()][:80])

print("\nMSOA share-like columns:")
print([c for c in msoa_csv.columns if "share" in c.lower()][:80])


Ward CSV: (7572, 164) k7_ward25_named_full_profile_v1_enriched.csv
MSOA CSV: (6856, 159) k7_msoa21_named_full_profile_v1.csv
OA base: (188880, 16)
Ward boundaries: (8405, 12)
MSOA boundaries: (7264, 9)

Ward CSV columns: ['LAD25CD', 'LAD25NM', 'WD25CD', 'WD25NM', 'population', 'oa_count', 'cluster_0_population', 'cluster_1_population', 'cluster_2_population', 'cluster_3_population', 'cluster_4_population', 'cluster_5_population', 'cluster_6_population', 'cluster_0_share', 'cluster_1_share', 'cluster_2_share', 'cluster_3_share', 'cluster_4_share', 'cluster_5_share', 'cluster_6_share', 'dominant_cluster', 'second_cluster', 'dominant_cluster_share', 'second_cluster_share', 'cluster_fragmentation_index', 'dominant_cluster_name', 'second_cluster_name', 'student_transient_youth_share', 'student_transient_youth_population', 'rooted_older_homeowners_share', 'rooted_older_homeowners_population', 'stable_suburban_professionals_share', 'stable_suburban_professionals_population', 'cosmopolitan_you

## 4. Standardise boundary join columns

The expected join keys are:

- Ward boundary key: `WD25CD`
- Ward CSV key: `WD25CD`
- MSOA boundary key: `MSOA21CD`
- MSOA CSV key: `MSOA21CD`

In [5]:
def standardise_code_col(gdf, target_col, possible_cols):
    gdf = gdf.copy()

    if target_col in gdf.columns:
        gdf[target_col] = gdf[target_col].astype(str).str.strip()
        return gdf

    normalised = {re.sub(r"[^A-Za-z0-9]", "", c).upper(): c for c in gdf.columns}

    for col in possible_cols:
        key = re.sub(r"[^A-Za-z0-9]", "", col).upper()
        if key in normalised:
            gdf = gdf.rename(columns={normalised[key]: target_col})
            gdf[target_col] = gdf[target_col].astype(str).str.strip()
            return gdf

    raise KeyError(f"Could not find or create {target_col}. Available columns: {gdf.columns.tolist()}")

ward_boundaries = standardise_code_col(
    ward_boundaries,
    "WD25CD",
    ["WD25CD", "WD25_CODE", "WD25 Code", "WD25CDO", "wd25cd"]
)

msoa_boundaries = standardise_code_col(
    msoa_boundaries,
    "MSOA21CD",
    ["MSOA21CD", "MSOA21_CODE", "MSOA21 Code", "msoa21cd"]
)

ward_csv["WD25CD"] = ward_csv["WD25CD"].astype(str).str.strip()
msoa_csv["MSOA21CD"] = msoa_csv["MSOA21CD"].astype(str).str.strip()
oa_base["MSOA21CD"] = oa_base["MSOA21CD"].astype(str).str.strip()
oa_base["LAD25NM"] = oa_base["LAD25NM"].astype(str).str.strip()

print("Ward boundary key unique:", ward_boundaries["WD25CD"].nunique())
print("Ward CSV key unique:", ward_csv["WD25CD"].nunique())
print("MSOA boundary key unique:", msoa_boundaries["MSOA21CD"].nunique())
print("MSOA CSV key unique:", msoa_csv["MSOA21CD"].nunique())

Ward boundary key unique: 8405
Ward CSV key unique: 7572
MSOA boundary key unique: 7264
MSOA CSV key unique: 6856


## 5. North West filter

Ward outputs already contain `LAD25NM`.

MSOA outputs often do not contain LAD25, so this notebook derives a population-weighted MSOA → LAD25 best-fit from the OA base table.

In [6]:
north_west_lads = [
    # Cheshire
    "Cheshire East", "Cheshire West and Chester", "Halton", "Warrington",

    # Cumbria
    "Cumberland", "Westmorland and Furness",

    # Greater Manchester
    "Bolton", "Bury", "Manchester", "Oldham", "Rochdale", "Salford",
    "Stockport", "Tameside", "Trafford", "Wigan",

    # Lancashire / unitary / districts
    "Blackburn with Darwen", "Blackpool", "Burnley", "Chorley", "Fylde",
    "Hyndburn", "Lancaster", "Pendle", "Preston", "Ribble Valley",
    "Rossendale", "South Ribble", "West Lancashire", "Wyre",

    # Merseyside
    "Knowsley", "Liverpool", "Sefton", "St. Helens", "Wirral",
]

# Ward filter.
if "LAD25NM" not in ward_csv.columns:
    raise KeyError("ward_csv does not contain LAD25NM. Re-run notebooks 07/08 v2 before mapping.")

ward_nw_csv = ward_csv[ward_csv["LAD25NM"].isin(north_west_lads)].copy()

# MSOA -> LAD25 best-fit from OA base using population.
msoa_lad25 = (
    oa_base
    .dropna(subset=["MSOA21CD", "LAD25CD", "LAD25NM", "population"])
    .groupby(["MSOA21CD", "LAD25CD", "LAD25NM"], as_index=False)
    .agg(population_in_lad=("population", "sum"))
    .sort_values(["MSOA21CD", "population_in_lad"], ascending=[True, False])
    .drop_duplicates("MSOA21CD")
)

msoa_nw_csv = msoa_csv.merge(
    msoa_lad25[["MSOA21CD", "LAD25CD", "LAD25NM"]],
    on="MSOA21CD",
    how="left",
    validate="one_to_one"
)

msoa_nw_csv = msoa_nw_csv[msoa_nw_csv["LAD25NM"].isin(north_west_lads)].copy()

print("North West ward rows:", len(ward_nw_csv))
print("North West MSOA rows:", len(msoa_nw_csv))
print("North West LADs in ward data:", sorted(ward_nw_csv["LAD25NM"].dropna().unique()))

North West ward rows: 825
North West MSOA rows: 932
North West LADs in ward data: ['Blackburn with Darwen', 'Blackpool', 'Bolton', 'Burnley', 'Bury', 'Cheshire East', 'Cheshire West and Chester', 'Chorley', 'Cumberland', 'Fylde', 'Halton', 'Hyndburn', 'Knowsley', 'Lancaster', 'Liverpool', 'Manchester', 'Oldham', 'Pendle', 'Preston', 'Ribble Valley', 'Rochdale', 'Rossendale', 'Salford', 'Sefton', 'South Ribble', 'St. Helens', 'Stockport', 'Tameside', 'Trafford', 'Warrington', 'West Lancashire', 'Westmorland and Furness', 'Wigan', 'Wirral', 'Wyre']


## 6. Join map data to boundaries

This produces two GeoDataFrames:

- `ward_nw_gdf`
- `msoa_nw_gdf`

In [7]:
# Merge boundaries to the filtered North West CSVs.
#
# Important: boundary files often already contain name columns such as WD25NM or MSOA21NM.
# When pandas/geopandas merges two tables that both contain the same non-key column,
# it suffixes them, e.g. LAD25NM_boundary / LAD25NM_data, and the plain LAD25NM column disappears.
# That is what caused the later KeyError: 'LAD25NM'.
#
# The coalesce step below restores canonical columns after the merge.

ward_nw_gdf = ward_boundaries.merge(
    ward_nw_csv,
    on="WD25CD",
    how="inner",
    validate="one_to_one",
    suffixes=("_boundary", "_data")
)

msoa_nw_gdf = msoa_boundaries.merge(
    msoa_nw_csv,
    on="MSOA21CD",
    how="inner",
    validate="one_to_one",
    suffixes=("_boundary", "_data")
)


def coalesce_after_merge(gdf, target_col, preferred_suffixes=("_data", "_csv", "", "_boundary", "_x", "_y")):
    """Restore a canonical column after a merge that may have created suffixed duplicates."""
    gdf = gdf.copy()

    candidate_cols = []
    for suffix in preferred_suffixes:
        candidate = f"{target_col}{suffix}" if suffix else target_col
        if candidate in gdf.columns and candidate not in candidate_cols:
            candidate_cols.append(candidate)

    if not candidate_cols:
        raise KeyError(
            f"Could not create {target_col}. No candidate columns found. "
            f"Available columns include: {list(gdf.columns)[:80]}"
        )

    # Start with the preferred source, then fill blanks from lower-priority candidates.
    combined = gdf[candidate_cols[0]]
    for col in candidate_cols[1:]:
        combined = combined.combine_first(gdf[col])

    gdf[target_col] = combined
    return gdf


# Restore canonical geography columns needed later.
for col in ["LAD25CD", "LAD25NM", "WD25NM"]:
    ward_nw_gdf = coalesce_after_merge(ward_nw_gdf, col)

for col in ["LAD25CD", "LAD25NM", "MSOA21NM"]:
    msoa_nw_gdf = coalesce_after_merge(msoa_nw_gdf, col)

# Use British National Grid for consistent UK plotting.
ward_nw_gdf = ward_nw_gdf.to_crs(epsg=27700)
msoa_nw_gdf = msoa_nw_gdf.to_crs(epsg=27700)

print("Ward NW GeoDataFrame:", ward_nw_gdf.shape)
print("MSOA NW GeoDataFrame:", msoa_nw_gdf.shape)

print("Missing ward geometry:", ward_nw_gdf.geometry.isna().sum())
print("Missing MSOA geometry:", msoa_nw_gdf.geometry.isna().sum())

print("Missing ward LAD25NM:", ward_nw_gdf["LAD25NM"].isna().sum())
print("Missing MSOA LAD25NM:", msoa_nw_gdf["LAD25NM"].isna().sum())

print("\nWard columns containing LAD25:")
print([c for c in ward_nw_gdf.columns if "LAD25" in c])

print("\nMSOA columns containing LAD25:")
print([c for c in msoa_nw_gdf.columns if "LAD25" in c])

Ward NW GeoDataFrame: (825, 178)
MSOA NW GeoDataFrame: (932, 170)
Missing ward geometry: 0
Missing MSOA geometry: 0
Missing ward LAD25NM: 0
Missing MSOA LAD25NM: 0

Ward columns containing LAD25:
['LAD25CD_boundary', 'LAD25NM_boundary', 'LAD25NMW', 'LAD25CD_data', 'LAD25NM_data', 'LAD25CD', 'LAD25NM']

MSOA columns containing LAD25:
['LAD25CD', 'LAD25NM']


## 7. Colours and plotting helpers

The cluster colours are deliberately muted so the maps are readable rather than visually painful.

In [8]:
CLUSTER_NAMES = {
    0: "Student & Transient Youth",
    1: "Rooted Older Homeowners",
    2: "Stable Suburban Professionals",
    3: "Cosmopolitan Young Professional Core",
    4: "Settled Working Families / Skilled Trades Suburbs",
    5: "Settled Diverse Urban Communities",
    6: "Post-Industrial Estates / Deprived Working Communities",
}

CLUSTER_COLORS = {
    0: "#c77d9b",  # muted pink
    1: "#7b5ea7",  # purple
    2: "#356aa0",  # blue
    3: "#5aa6c8",  # light blue
    4: "#6a994e",  # muted green
    5: "#2a9d8f",  # teal green
    6: "#b4574d",  # muted red/rust
}

HEATMAP_COLUMNS = {
    "Post-Industrial Estates / Deprived Working Communities": [
        "post_industrial_estates_deprived_working_communities_share",
        "Post-Industrial Estates / Deprived Working Communities_share",
        "cluster_6_share",
    ],
    "Settled Working Families / Skilled Trades Suburbs": [
        "settled_working_families_skilled_trades_suburbs_share",
        "Settled Working Families / Skilled Trades Suburbs_share",
        "cluster_4_share",
    ],
    "Rooted Older Homeowners": [
        "rooted_older_homeowners_share",
        "Rooted Older Homeowners_share",
        "cluster_1_share",
    ],
    "Student & Transient Youth": [
        "student_transient_youth_share",
        "Student & Transient Youth_share",
        "cluster_0_share",
    ],
    "Cosmopolitan Young Professional Core": [
        "cosmopolitan_young_professional_core_share",
        "Cosmopolitan Young Professional Core_share",
        "cluster_3_share",
    ],
}


def safe_filename(text):
    text = re.sub(r"[^A-Za-z0-9]+", "_", text).strip("_").lower()
    return text


def normalise_col_name(text):
    return re.sub(r"[^A-Za-z0-9]+", "_", str(text)).strip("_").lower()


def first_existing_col(df, candidates):
    """
    Return the first matching column.

    It tries:
    1. exact match
    2. normalised match, so columns with punctuation/spaces still match
    """
    columns = list(df.columns)

    for col in candidates:
        if col in columns:
            return col

    normalised_lookup = {normalise_col_name(c): c for c in columns}

    for col in candidates:
        key = normalise_col_name(col)
        if key in normalised_lookup:
            return normalised_lookup[key]

    return None


def debug_heatmap_columns(gdf, label, candidates):
    found = first_existing_col(gdf, candidates)
    if found is None:
        print(f"Column not found for {label}")
        print("Candidates tried:", candidates)
        print("Available share-like columns:")
        print([c for c in gdf.columns if "share" in c.lower()])
    return found


def plot_categorical_clusters(gdf, title, output_path, figsize=(14, 12)):
    plot_gdf = gdf.copy()
    plot_gdf["_cluster_color"] = plot_gdf["dominant_cluster"].map(CLUSTER_COLORS).fillna("#d9d9d9")

    fig, ax = plt.subplots(figsize=figsize)
    plot_gdf.plot(
        ax=ax,
        color=plot_gdf["_cluster_color"],
        edgecolor="white",
        linewidth=0.05,
    )

    legend_handles = [
        Patch(facecolor=CLUSTER_COLORS[cid], edgecolor="none", label=f"{cid}: {name}")
        for cid, name in CLUSTER_NAMES.items()
    ]

    ax.legend(
        handles=legend_handles,
        title="Dominant cluster",
        loc="lower left",
        frameon=True,
        fontsize=8,
        title_fontsize=9,
    )

    ax.set_title(title, fontsize=16, pad=12)
    ax.set_axis_off()
    fig.tight_layout()
    fig.savefig(output_path, dpi=220, bbox_inches="tight")
    plt.close(fig)
    print("Saved:", output_path.name)


def plot_continuous(gdf, column, title, output_path, cmap="viridis", figsize=(14, 12), vmin=None, vmax=None):
    fig, ax = plt.subplots(figsize=figsize)

    gdf.plot(
        ax=ax,
        column=column,
        cmap=cmap,
        linewidth=0.05,
        edgecolor="white",
        legend=True,
        vmin=vmin,
        vmax=vmax,
        missing_kwds={"color": "#eeeeee", "label": "Missing"},
    )

    ax.set_title(title, fontsize=16, pad=12)
    ax.set_axis_off()
    fig.tight_layout()
    fig.savefig(output_path, dpi=220, bbox_inches="tight")
    plt.close(fig)
    print("Saved:", output_path.name)


## 8. Dominant cluster maps

These are the main atlas maps.

In [9]:
plot_categorical_clusters(
    ward_nw_gdf,
    title="North West Ward25 — Dominant K=7 Cluster",
    output_path=MAP_DIR / "k7_nw_ward25_dominant_cluster_v1.png",
)

plot_categorical_clusters(
    msoa_nw_gdf,
    title="North West MSOA21 — Dominant K=7 Cluster",
    output_path=MAP_DIR / "k7_nw_msoa21_dominant_cluster_v1.png",
)

Saved: k7_nw_ward25_dominant_cluster_v1.png
Saved: k7_nw_msoa21_dominant_cluster_v1.png


## 9. Confidence and fragmentation maps

`dominant_cluster_share` shows how dominant the leading cluster is in each geography.

`cluster_fragmentation_index` shows how mixed the area is. Higher values mean more mixed.

In [10]:
plot_continuous(
    ward_nw_gdf,
    column="dominant_cluster_share",
    title="North West Ward25 — Dominant Cluster Share / Confidence",
    output_path=MAP_DIR / "k7_nw_ward25_dominant_cluster_share_v1.png",
    cmap="YlGnBu",
    vmin=0,
    vmax=1,
)

plot_continuous(
    ward_nw_gdf,
    column="cluster_fragmentation_index",
    title="North West Ward25 — Cluster Fragmentation Index",
    output_path=MAP_DIR / "k7_nw_ward25_fragmentation_index_v1.png",
    cmap="magma",
)

plot_continuous(
    msoa_nw_gdf,
    column="dominant_cluster_share",
    title="North West MSOA21 — Dominant Cluster Share / Confidence",
    output_path=MAP_DIR / "k7_nw_msoa21_dominant_cluster_share_v1.png",
    cmap="YlGnBu",
    vmin=0,
    vmax=1,
)

plot_continuous(
    msoa_nw_gdf,
    column="cluster_fragmentation_index",
    title="North West MSOA21 — Cluster Fragmentation Index",
    output_path=MAP_DIR / "k7_nw_msoa21_fragmentation_index_v1.png",
    cmap="magma",
)

Saved: k7_nw_ward25_dominant_cluster_share_v1.png
Saved: k7_nw_ward25_fragmentation_index_v1.png
Saved: k7_nw_msoa21_dominant_cluster_share_v1.png
Saved: k7_nw_msoa21_fragmentation_index_v1.png


## 10. Individual cluster-share heatmaps

These show the intensity of selected clusters across the North West.

The notebook produces both Ward25 and MSOA21 versions where the required share columns exist.

In [11]:
for label, candidates in HEATMAP_COLUMNS.items():
    ward_col = debug_heatmap_columns(ward_nw_gdf, f"Ward — {label}", candidates)
    msoa_col = debug_heatmap_columns(msoa_nw_gdf, f"MSOA — {label}", candidates)

    slug = safe_filename(label)

    if ward_col is not None:
        plot_continuous(
            ward_nw_gdf,
            column=ward_col,
            title=f"North West Ward25 — {label} Share",
            output_path=MAP_DIR / f"k7_nw_ward25_heatmap_{slug}_v1.png",
            cmap="YlOrRd",
            vmin=0,
            vmax=1,
        )

    if msoa_col is not None:
        plot_continuous(
            msoa_nw_gdf,
            column=msoa_col,
            title=f"North West MSOA21 — {label} Share",
            output_path=MAP_DIR / f"k7_nw_msoa21_heatmap_{slug}_v1.png",
            cmap="YlOrRd",
            vmin=0,
            vmax=1,
        )


Saved: k7_nw_ward25_heatmap_post_industrial_estates_deprived_working_communities_v1.png
Saved: k7_nw_msoa21_heatmap_post_industrial_estates_deprived_working_communities_v1.png
Saved: k7_nw_ward25_heatmap_settled_working_families_skilled_trades_suburbs_v1.png
Saved: k7_nw_msoa21_heatmap_settled_working_families_skilled_trades_suburbs_v1.png
Saved: k7_nw_ward25_heatmap_rooted_older_homeowners_v1.png
Saved: k7_nw_msoa21_heatmap_rooted_older_homeowners_v1.png
Saved: k7_nw_ward25_heatmap_student_transient_youth_v1.png
Saved: k7_nw_msoa21_heatmap_student_transient_youth_v1.png
Saved: k7_nw_ward25_heatmap_cosmopolitan_young_professional_core_v1.png
Saved: k7_nw_msoa21_heatmap_cosmopolitan_young_professional_core_v1.png


## 11. Regional map outputs

This section produces the same core maps for each North West sub-region:

- Cheshire
- Cumbria
- Greater Manchester
- Lancashire
- Merseyside

For each region it creates Ward25 and MSOA21 versions of:

1. Dominant cluster
2. Dominant cluster share / confidence
3. Fragmentation index
4. Selected individual cluster-share heatmaps

Outputs are saved under:

```text
data/processed/maps_v1/by_region/<region_slug>/
```

Note: Sefton is included in Merseyside here because the geography is Ward25. Treat Sefton maps as provisional for 2026/2027 work until newer boundaries exist.

In [12]:
# North West sub-region definitions by LAD25 name.
# Add aliases where ONS spelling can vary, e.g. St. Helens vs St Helens.
NW_REGION_BY_LAD = {
    # Cheshire
    "Cheshire East": "Cheshire",
    "Cheshire West and Chester": "Cheshire",
    "Halton": "Cheshire",
    "Warrington": "Cheshire",

    # Cumbria
    "Cumberland": "Cumbria",
    "Westmorland and Furness": "Cumbria",

    # Greater Manchester
    "Bolton": "Greater Manchester",
    "Bury": "Greater Manchester",
    "Manchester": "Greater Manchester",
    "Oldham": "Greater Manchester",
    "Rochdale": "Greater Manchester",
    "Salford": "Greater Manchester",
    "Stockport": "Greater Manchester",
    "Tameside": "Greater Manchester",
    "Trafford": "Greater Manchester",
    "Wigan": "Greater Manchester",

    # Lancashire, including unitary authorities
    "Blackburn with Darwen": "Lancashire",
    "Blackpool": "Lancashire",
    "Burnley": "Lancashire",
    "Chorley": "Lancashire",
    "Fylde": "Lancashire",
    "Hyndburn": "Lancashire",
    "Lancaster": "Lancashire",
    "Pendle": "Lancashire",
    "Preston": "Lancashire",
    "Ribble Valley": "Lancashire",
    "Rossendale": "Lancashire",
    "South Ribble": "Lancashire",
    "West Lancashire": "Lancashire",
    "Wyre": "Lancashire",

    # Merseyside
    "Knowsley": "Merseyside",
    "Liverpool": "Merseyside",
    "Sefton": "Merseyside",
    "St. Helens": "Merseyside",
    "St Helens": "Merseyside",
    "Wirral": "Merseyside",
}

REGION_ORDER = [
    "Cheshire",
    "Cumbria",
    "Greater Manchester",
    "Lancashire",
    "Merseyside",
]

REGION_MAP_DIR = MAP_DIR / "by_region"
REGION_MAP_DIR.mkdir(parents=True, exist_ok=True)

ward_nw_gdf = ward_nw_gdf.copy()
msoa_nw_gdf = msoa_nw_gdf.copy()

# Defensive check: these columns should have been restored immediately after the boundary merge.
for label, gdf in [("ward", ward_nw_gdf), ("msoa", msoa_nw_gdf)]:
    if "LAD25NM" not in gdf.columns:
        raise KeyError(
            f"{label}_nw_gdf does not contain LAD25NM. "
            "Re-run the merge cell that restores canonical columns after suffixing. "
            f"Available columns: {gdf.columns.tolist()}"
        )

ward_nw_gdf["LAD25NM"] = ward_nw_gdf["LAD25NM"].astype(str).str.strip()
msoa_nw_gdf["LAD25NM"] = msoa_nw_gdf["LAD25NM"].astype(str).str.strip()

ward_nw_gdf["nw_region"] = ward_nw_gdf["LAD25NM"].map(NW_REGION_BY_LAD)
msoa_nw_gdf["nw_region"] = msoa_nw_gdf["LAD25NM"].map(NW_REGION_BY_LAD)

print("Ward region coverage:")
display(
    ward_nw_gdf
    .groupby("nw_region", dropna=False)
    .agg(
        wards=("WD25CD", "nunique"),
        population=("population", "sum"),
    )
    .reset_index()
)

print("MSOA region coverage:")
display(
    msoa_nw_gdf
    .groupby("nw_region", dropna=False)
    .agg(
        msoas=("MSOA21CD", "nunique"),
        population=("population", "sum"),
    )
    .reset_index()
)

# Flag any unmatched LAD names early.
unmatched_ward_lads = sorted(ward_nw_gdf.loc[ward_nw_gdf["nw_region"].isna(), "LAD25NM"].dropna().unique())
unmatched_msoa_lads = sorted(msoa_nw_gdf.loc[msoa_nw_gdf["nw_region"].isna(), "LAD25NM"].dropna().unique())

if unmatched_ward_lads:
    print("Unmatched ward LAD names:", unmatched_ward_lads)
if unmatched_msoa_lads:
    print("Unmatched MSOA LAD names:", unmatched_msoa_lads)

Ward region coverage:


,nw_region,wards,population
0,Cheshire,137,1095365
1,Cumbria,79,499865
2,Greater Manchester,215,2867794
3,Lancashire,253,1531186
4,Merseyside,141,1423339


MSOA region coverage:


,nw_region,msoas,population
0,Cheshire,140,1095365
1,Cumbria,64,499865
2,Greater Manchester,353,2867794
3,Lancashire,190,1531186
4,Merseyside,185,1423339


## 12. Create regional maps

This loop creates a complete set of region-specific PNG maps. It skips a region/layer if no rows are available.

In [13]:
def plot_region_set(gdf, level_label, code_col, region_name, region_dir):
    """Create all standard maps for one region and one geography level."""
    region_slug = safe_filename(region_name)
    level_slug = safe_filename(level_label)

    region_gdf = gdf[gdf["nw_region"].eq(region_name)].copy()

    if region_gdf.empty:
        print(f"Skipping {region_name} / {level_label}: no rows")
        return

    # Dominant cluster map.
    plot_categorical_clusters(
        region_gdf,
        title=f"{region_name} {level_label} — Dominant K=7 Cluster",
        output_path=region_dir / f"k7_{region_slug}_{level_slug}_dominant_cluster_v1.png",
        figsize=(11, 10),
    )

    # Dominant cluster confidence.
    if "dominant_cluster_share" in region_gdf.columns:
        plot_continuous(
            region_gdf,
            column="dominant_cluster_share",
            title=f"{region_name} {level_label} — Dominant Cluster Share / Confidence",
            output_path=region_dir / f"k7_{region_slug}_{level_slug}_dominant_cluster_share_v1.png",
            cmap="YlGnBu",
            figsize=(11, 10),
            vmin=0,
            vmax=1,
        )

    # Fragmentation.
    if "cluster_fragmentation_index" in region_gdf.columns:
        plot_continuous(
            region_gdf,
            column="cluster_fragmentation_index",
            title=f"{region_name} {level_label} — Cluster Fragmentation Index",
            output_path=region_dir / f"k7_{region_slug}_{level_slug}_fragmentation_index_v1.png",
            cmap="magma_r",
            figsize=(11, 10),
            vmin=0,
            vmax=1,
        )

    # Individual cluster heatmaps.
    for label, candidates in HEATMAP_COLUMNS.items():
        heat_col = debug_heatmap_columns(region_gdf, f"{region_name} {level_label} — {label}", candidates)
        if heat_col is None:
            continue

        heat_slug = safe_filename(label)

        plot_continuous(
            region_gdf,
            column=heat_col,
            title=f"{region_name} {level_label} — {label} Share",
            output_path=region_dir / f"k7_{region_slug}_{level_slug}_heatmap_{heat_slug}_v1.png",
            cmap="YlOrRd",
            figsize=(11, 10),
            vmin=0,
            vmax=1,
        )

    # Save the joined regional layer as a GeoPackage too.
    gpkg_path = region_dir / f"k7_{region_slug}_{level_slug}_joined_map_layer_v1.gpkg"
    region_gdf.to_file(gpkg_path, layer=f"{level_slug}_k7", driver="GPKG")
    print("Saved:", gpkg_path.name)


for region_name in REGION_ORDER:
    region_slug = safe_filename(region_name)
    region_dir = REGION_MAP_DIR / region_slug
    region_dir.mkdir(parents=True, exist_ok=True)

    print("\n" + "=" * 80)
    print("Creating regional maps for:", region_name)

    plot_region_set(
        ward_nw_gdf,
        level_label="Ward25",
        code_col="WD25CD",
        region_name=region_name,
        region_dir=region_dir,
    )

    plot_region_set(
        msoa_nw_gdf,
        level_label="MSOA21",
        code_col="MSOA21CD",
        region_name=region_name,
        region_dir=region_dir,
    )


Creating regional maps for: Cheshire
Saved: k7_cheshire_ward25_dominant_cluster_v1.png
Saved: k7_cheshire_ward25_dominant_cluster_share_v1.png
Saved: k7_cheshire_ward25_fragmentation_index_v1.png
Saved: k7_cheshire_ward25_heatmap_post_industrial_estates_deprived_working_communities_v1.png
Saved: k7_cheshire_ward25_heatmap_settled_working_families_skilled_trades_suburbs_v1.png
Saved: k7_cheshire_ward25_heatmap_rooted_older_homeowners_v1.png
Saved: k7_cheshire_ward25_heatmap_student_transient_youth_v1.png
Saved: k7_cheshire_ward25_heatmap_cosmopolitan_young_professional_core_v1.png
Saved: k7_cheshire_ward25_joined_map_layer_v1.gpkg
Saved: k7_cheshire_msoa21_dominant_cluster_v1.png
Saved: k7_cheshire_msoa21_dominant_cluster_share_v1.png
Saved: k7_cheshire_msoa21_fragmentation_index_v1.png
Saved: k7_cheshire_msoa21_heatmap_post_industrial_estates_deprived_working_communities_v1.png
Saved: k7_cheshire_msoa21_heatmap_settled_working_families_skilled_trades_suburbs_v1.png
Saved: k7_cheshire_

## 13. Regional output list

This prints every generated regional map/layer so you can quickly check which files were created.

In [14]:
print("Generated regional maps and geospatial files:")
for path in sorted(REGION_MAP_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(MAP_DIR))

Generated regional maps and geospatial files:
by_region\cheshire\k7_cheshire_msoa21_dominant_cluster_share_v1.png
by_region\cheshire\k7_cheshire_msoa21_dominant_cluster_v1.png
by_region\cheshire\k7_cheshire_msoa21_fragmentation_index_v1.png
by_region\cheshire\k7_cheshire_msoa21_heatmap_cosmopolitan_young_professional_core_v1.png
by_region\cheshire\k7_cheshire_msoa21_heatmap_post_industrial_estates_deprived_working_communities_v1.png
by_region\cheshire\k7_cheshire_msoa21_heatmap_rooted_older_homeowners_v1.png
by_region\cheshire\k7_cheshire_msoa21_heatmap_settled_working_families_skilled_trades_suburbs_v1.png
by_region\cheshire\k7_cheshire_msoa21_heatmap_student_transient_youth_v1.png
by_region\cheshire\k7_cheshire_msoa21_joined_map_layer_v1.gpkg
by_region\cheshire\k7_cheshire_ward25_dominant_cluster_share_v1.png
by_region\cheshire\k7_cheshire_ward25_dominant_cluster_v1.png
by_region\cheshire\k7_cheshire_ward25_fragmentation_index_v1.png
by_region\cheshire\k7_cheshire_ward25_heatmap_cosm

## 14. LAD-level map outputs

This section creates a separate folder for each North West LAD/council area.

For each LAD it creates both Ward25 and MSOA21 versions of:

- dominant cluster
- dominant cluster share / confidence
- fragmentation index
- individual cluster-share heatmaps
- joined GeoPackage layer

Output location:

```text
data/processed/maps_v1/by_lad/<lad_name_slug>/
```

This is intentionally verbose: it gives you a clear atlas pack for every council area, suitable for later review, reporting and targeted council-level analysis.

In [15]:
# Create a complete map pack for every LAD/council in the North West.
#
# This may create a lot of files:
# roughly 2 geography levels × 8 maps/layers × ~35 LADs.
# Set RUN_LAD_MAPS = False if you only want to run the regional/North West maps.

RUN_LAD_MAPS = True
RUN_LAD_GPKG = True

# Optional: restrict to a subset while testing, e.g.
# LAD_MAPS_TO_CREATE = ["Preston", "Blackpool"]
LAD_MAPS_TO_CREATE = None

LAD_MAP_DIR = MAP_DIR / "by_lad"
LAD_MAP_DIR.mkdir(parents=True, exist_ok=True)


def plot_lad_set(gdf, level_label, code_col, lad_name, lad_dir):
    """Create all standard maps for one LAD and one geography level."""
    lad_slug = safe_filename(lad_name)
    level_slug = safe_filename(level_label)

    lad_gdf = gdf[gdf["LAD25NM"].eq(lad_name)].copy()

    if lad_gdf.empty:
        print(f"Skipping {lad_name} / {level_label}: no rows")
        return

    # Dominant cluster map.
    plot_categorical_clusters(
        lad_gdf,
        title=f"{lad_name} {level_label} — Dominant K=7 Cluster",
        output_path=lad_dir / f"k7_{lad_slug}_{level_slug}_dominant_cluster_v1.png",
        figsize=(10, 10),
    )

    # Dominant cluster confidence.
    if "dominant_cluster_share" in lad_gdf.columns:
        plot_continuous(
            lad_gdf,
            column="dominant_cluster_share",
            title=f"{lad_name} {level_label} — Dominant Cluster Share / Confidence",
            output_path=lad_dir / f"k7_{lad_slug}_{level_slug}_dominant_cluster_share_v1.png",
            cmap="YlGnBu",
            figsize=(10, 10),
            vmin=0,
            vmax=1,
        )

    # Fragmentation index.
    if "cluster_fragmentation_index" in lad_gdf.columns:
        plot_continuous(
            lad_gdf,
            column="cluster_fragmentation_index",
            title=f"{lad_name} {level_label} — Cluster Fragmentation Index",
            output_path=lad_dir / f"k7_{lad_slug}_{level_slug}_fragmentation_index_v1.png",
            cmap="magma_r",
            figsize=(10, 10),
            vmin=0,
            vmax=1,
        )

    # Individual cluster heatmaps.
    for label, candidates in HEATMAP_COLUMNS.items():
        heat_col = debug_heatmap_columns(
            lad_gdf,
            f"{lad_name} {level_label} — {label}",
            candidates
        )

        if heat_col is None:
            continue

        heat_slug = safe_filename(label)

        plot_continuous(
            lad_gdf,
            column=heat_col,
            title=f"{lad_name} {level_label} — {label} Share",
            output_path=lad_dir / f"k7_{lad_slug}_{level_slug}_heatmap_{heat_slug}_v1.png",
            cmap="YlOrRd",
            figsize=(10, 10),
            vmin=0,
            vmax=1,
        )

    if RUN_LAD_GPKG:
        gpkg_path = lad_dir / f"k7_{lad_slug}_{level_slug}_joined_map_layer_v1.gpkg"
        lad_gdf.to_file(gpkg_path, layer=f"{level_slug}_k7", driver="GPKG")
        print("Saved:", gpkg_path.name)


if RUN_LAD_MAPS:
    # Use the ward GeoDataFrame as the canonical LAD list.
    # This avoids creating MSOA-only LAD folders if a council has no ward rows for some reason.
    lad_names = sorted(ward_nw_gdf["LAD25NM"].dropna().unique())

    if LAD_MAPS_TO_CREATE is not None:
        lad_names = [lad for lad in lad_names if lad in LAD_MAPS_TO_CREATE]

    print("LAD map packs to create:", len(lad_names))
    print(lad_names)

    for lad_name in lad_names:
        lad_slug = safe_filename(lad_name)
        lad_dir = LAD_MAP_DIR / lad_slug
        lad_dir.mkdir(parents=True, exist_ok=True)

        print("\n" + "=" * 80)
        print("Creating LAD maps for:", lad_name)

        plot_lad_set(
            ward_nw_gdf,
            level_label="Ward25",
            code_col="WD25CD",
            lad_name=lad_name,
            lad_dir=lad_dir,
        )

        plot_lad_set(
            msoa_nw_gdf,
            level_label="MSOA21",
            code_col="MSOA21CD",
            lad_name=lad_name,
            lad_dir=lad_dir,
        )
else:
    print("LAD map generation skipped. Set RUN_LAD_MAPS = True to run.")

LAD map packs to create: 35
['Blackburn with Darwen', 'Blackpool', 'Bolton', 'Burnley', 'Bury', 'Cheshire East', 'Cheshire West and Chester', 'Chorley', 'Cumberland', 'Fylde', 'Halton', 'Hyndburn', 'Knowsley', 'Lancaster', 'Liverpool', 'Manchester', 'Oldham', 'Pendle', 'Preston', 'Ribble Valley', 'Rochdale', 'Rossendale', 'Salford', 'Sefton', 'South Ribble', 'St. Helens', 'Stockport', 'Tameside', 'Trafford', 'Warrington', 'West Lancashire', 'Westmorland and Furness', 'Wigan', 'Wirral', 'Wyre']

Creating LAD maps for: Blackburn with Darwen
Saved: k7_blackburn_with_darwen_ward25_dominant_cluster_v1.png
Saved: k7_blackburn_with_darwen_ward25_dominant_cluster_share_v1.png
Saved: k7_blackburn_with_darwen_ward25_fragmentation_index_v1.png
Saved: k7_blackburn_with_darwen_ward25_heatmap_post_industrial_estates_deprived_working_communities_v1.png
Saved: k7_blackburn_with_darwen_ward25_heatmap_settled_working_families_skilled_trades_suburbs_v1.png
Saved: k7_blackburn_with_darwen_ward25_heatmap_r

## 15. LAD output list

This prints the LAD-specific folders and files generated under `maps_v1/by_lad`.

In [16]:
if LAD_MAP_DIR.exists():
    print("Generated LAD maps and geospatial files:")
    for path in sorted(LAD_MAP_DIR.rglob("*")):
        if path.is_file():
            print(path.relative_to(MAP_DIR))
else:
    print("No LAD map directory found.")

Generated LAD maps and geospatial files:
by_lad\blackburn_with_darwen\k7_blackburn_with_darwen_msoa21_dominant_cluster_share_v1.png
by_lad\blackburn_with_darwen\k7_blackburn_with_darwen_msoa21_dominant_cluster_v1.png
by_lad\blackburn_with_darwen\k7_blackburn_with_darwen_msoa21_fragmentation_index_v1.png
by_lad\blackburn_with_darwen\k7_blackburn_with_darwen_msoa21_heatmap_cosmopolitan_young_professional_core_v1.png
by_lad\blackburn_with_darwen\k7_blackburn_with_darwen_msoa21_heatmap_post_industrial_estates_deprived_working_communities_v1.png
by_lad\blackburn_with_darwen\k7_blackburn_with_darwen_msoa21_heatmap_rooted_older_homeowners_v1.png
by_lad\blackburn_with_darwen\k7_blackburn_with_darwen_msoa21_heatmap_settled_working_families_skilled_trades_suburbs_v1.png
by_lad\blackburn_with_darwen\k7_blackburn_with_darwen_msoa21_heatmap_student_transient_youth_v1.png
by_lad\blackburn_with_darwen\k7_blackburn_with_darwen_msoa21_joined_map_layer_v1.gpkg
by_lad\blackburn_with_darwen\k7_blackburn_w

## 11. Save joined GeoPackages

These are optional but useful. They let you open the joined map layers directly in QGIS without rejoining CSVs.

In [17]:
ward_gpkg_path = MAP_DIR / "k7_nw_ward25_joined_map_layer_v1.gpkg"
msoa_gpkg_path = MAP_DIR / "k7_nw_msoa21_joined_map_layer_v1.gpkg"

ward_nw_gdf.to_file(ward_gpkg_path, layer="ward25_k7", driver="GPKG")
msoa_nw_gdf.to_file(msoa_gpkg_path, layer="msoa21_k7", driver="GPKG")

print("Saved:", ward_gpkg_path.name)
print("Saved:", msoa_gpkg_path.name)

Saved: k7_nw_ward25_joined_map_layer_v1.gpkg
Saved: k7_nw_msoa21_joined_map_layer_v1.gpkg


## 12. Final output list

In [18]:
print("Generated maps and geospatial files:")
for path in sorted(MAP_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(MAP_DIR))

Generated maps and geospatial files:
by_lad\blackburn_with_darwen\k7_blackburn_with_darwen_msoa21_dominant_cluster_share_v1.png
by_lad\blackburn_with_darwen\k7_blackburn_with_darwen_msoa21_dominant_cluster_v1.png
by_lad\blackburn_with_darwen\k7_blackburn_with_darwen_msoa21_fragmentation_index_v1.png
by_lad\blackburn_with_darwen\k7_blackburn_with_darwen_msoa21_heatmap_cosmopolitan_young_professional_core_v1.png
by_lad\blackburn_with_darwen\k7_blackburn_with_darwen_msoa21_heatmap_post_industrial_estates_deprived_working_communities_v1.png
by_lad\blackburn_with_darwen\k7_blackburn_with_darwen_msoa21_heatmap_rooted_older_homeowners_v1.png
by_lad\blackburn_with_darwen\k7_blackburn_with_darwen_msoa21_heatmap_settled_working_families_skilled_trades_suburbs_v1.png
by_lad\blackburn_with_darwen\k7_blackburn_with_darwen_msoa21_heatmap_student_transient_youth_v1.png
by_lad\blackburn_with_darwen\k7_blackburn_with_darwen_msoa21_joined_map_layer_v1.gpkg
by_lad\blackburn_with_darwen\k7_blackburn_with_